In [1]:
import torch.nn as nn
import torch
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchmetrics.text.rouge import ROUGEScore
from torch.nn.utils.rnn import pad_sequence
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 
import math
from tqdm.auto import tqdm
import os
import json
from copy import deepcopy
from dataclasses import dataclass
from tokenizers import Tokenizer
import random

## Архитектура Трансформера

In [2]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, h, dropout_p=0.1):
        super().__init__()
        self.d_model = d_model
        self.h = h
        self.d_k = d_model // h

        self.W_O = nn.Parameter(
            nn.init.xavier_normal_(
                torch.randn(d_model, d_model)
            )
        )
        
        self.W_Q = nn.Parameter(
            nn.init.xavier_normal_(
                torch.randn(d_model, d_model)
            )
        )
       
        self.W_K = nn.Parameter(
            nn.init.xavier_normal_(
                torch.randn(d_model, d_model)
            )
        )
    
        self.W_V = nn.Parameter(
            nn.init.xavier_normal_(
                torch.randn(d_model, d_model)
            )
        )

        self.dropout = nn.Dropout(dropout_p)
        self.kv_cache = None
        
    def forward(self, Q, K, V, mask=None, cache=False):
        """
        Q:     [Batch, Seq_len1, d_model]
        K, V:  [Batch, Seq_len2, d_model]
        mask:  [Batch, Seq_len1, Seq_len2]
        """
            
        Q_heads = Q @ self.W_Q    # [Batch, Seq_len1, d_model]
        K_heads = K @ self.W_K    # [Batch, Seq_len2, d_model]
        V_heads = V @ self.W_V    # [Batch, Seq_len2, d_model]

        if cache:
            if self.kv_cache is not None:
                K_cached = torch.concat([self.kv_cache[0], K_heads], dim=1)
                V_cached = torch.concat([self.kv_cache[1], V_heads], dim=1)
            else:
                K_cached = K_heads
                V_cached = V_heads
                
            self.kv_cache = (K_cached, V_cached)
            K_heads = K_cached
            V_heads = V_cached
        
        # [Batch, Seq_len, d_model] -> [Batch, Seq_len, h, d_k] -> [Batch, h, Seq_len, d_k]
        Q_heads = Q_heads.view(Q_heads.shape[0], Q_heads.shape[1], self.h, self.d_k).transpose(1, 2)
        K_heads = K_heads.view(K_heads.shape[0], K_heads.shape[1], self.h, self.d_k).transpose(1, 2)
        V_heads = V_heads.view(V_heads.shape[0], V_heads.shape[1], self.h, self.d_k).transpose(1, 2)
        
        # [Batch, h, Seq_len1, Seq_len2]
        attention_maps = torch.matmul(Q_heads, K_heads.transpose(-1, -2)) / math.sqrt(self.d_k)
        if mask is not None:
            mask = mask.unsqueeze(1)                               # [Batch, 1, Seq_len1, Seq_len2]
            attention_maps.masked_fill_(mask == 0, -1e9)
            
        attention_weights = torch.softmax(attention_maps, dim=-1)  # [Batch, h, Seq_len1, Seq_len2]
        attention_weights = self.dropout(attention_weights)
        
        scores = torch.matmul(attention_weights, V_heads)          # [Batch, h, Seq_len1, d_k]
        scores = scores.transpose(1, 2).contiguous()               # [Batch, Seq_len1, h, d_k]
        scores = scores.view(scores.shape[0],                      # [Batch, Seq_len1, d_model]
                             scores.shape[1], 
                             self.d_model)
        
        result = scores @ self.W_O                                 # [Batch, Seq_len1, d_model]  
        
        return result

    def clear_cache(self):
        self.kv_cache = None
    
    
class LayerNorm(nn.Module):
    def __init__(self, size):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(size))
        self.bias = nn.Parameter(torch.zeros(size))
        
    def forward(self, X, eps=1e-6):
        """
        X: [Batch, Seq_len, size]
        """
        mean = X.mean(dim=-1, keepdim=True)              # [Batch, Seq_len, 1]  [1] в конце благодаря keepdim=True
        var = X.var(dim=-1, correction=0, keepdim=True)  # [Batch, Seq_len, 1]  который сохраняет размерность
        
        return self.gamma * (X - mean) / torch.sqrt(var + eps) + self.bias
    

class FeedForwardLayers(nn.Module):
    def __init__(self, d_model, dropout_p=0.1):
        super().__init__()
        self.d_model = d_model
        self.layer1 = nn.Linear(d_model, 4 * d_model)
        self.layer2 = nn.Linear(4 * d_model, d_model)
        self.act = nn.GELU()
        self.dropout = nn.Dropout(dropout_p)
        
    def forward(self, X):
        """
        X: [Batch, Seq_len, d_model]
        """
        out = self.act(self.layer1(X))
        out = self.dropout(out)
        out = self.layer2(out)
        
        return out
    

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_len):
        super().__init__()
        self.d_model = d_model
        self.max_seq_len = max_seq_len
        
        pe = torch.zeros(max_seq_len, d_model)
        pos = torch.arange(0, max_seq_len).unsqueeze(1).expand(max_seq_len, d_model // 2)  # [max_seq_len, d_model // 2]
        i = torch.arange(0, d_model//2).unsqueeze(0).expand(max_seq_len, d_model//2)       # [max_seq_len, d_model // 2]
        arg = pos / 10000**(2 * i / d_model)
        pe[:, ::2] = torch.sin(arg)
        pe[:, 1::2] = torch.cos(arg)
        
        pe = pe.unsqueeze(0)            # [1, max_seq_len, d_model]
        self.register_buffer("pe", pe)
        
    def forward(self, X):
        """
        X: [Batch, Seq_len, d_model] or int
        """
        if isinstance(X, int):
            out = self.pe[:, X, :]
        else:
            out = X + self.pe[:, :X.shape[1], :]
        return out


class Embedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)

    def forward(self, X):
        out = self.embedding(X) * math.sqrt(self.d_model)
        return out

In [3]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, h, dropout_p=0.1):
        super().__init__()
        self.mha = MultiHeadAttention(d_model, h, dropout_p)
        self.norm1 = LayerNorm(d_model)
        self.ff = FeedForwardLayers(d_model, dropout_p)
        self.norm2 = LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout_p)
        
    def forward(self, X, padding_mask=None):
        """
        X:            [Batch, Seq_len, d_model]
        padding_mask: [Batch, 1, Seq_len]
        """
        out = self.norm1(X)                                # [Batch, Seq_len, d_model]
        out = self.mha(out, out, out, mask=padding_mask)   # [Batch, Seq_len, d_model]
        out = self.dropout(out)
        out = out + X

        norm_out = self.norm2(out)                         # [Batch, Seq_len, d_model]
        ff_out = self.ff(norm_out)                         # [Batch, Seq_len, d_model]
        ff_out = self.dropout(ff_out)
        out = ff_out + out      
        
        return out
    

class Encoder(nn.Module):
    def __init__(self, d_model, h, num_layers, dropout_p=0.1):
        super().__init__()
        self.num_layers = num_layers
        self.layers = nn.ModuleList(
            EncoderLayer(d_model, h, dropout_p) for _ in range(num_layers)
        )
        self.norm = LayerNorm(d_model)
        
    def forward(self, X, padding_mask=None):
        """
        X:            [Batch, Seq_len, d_model]
        padding_mask: [Batch, 1, Seq_len]
        """
        for layer in self.layers:
            X = layer(X, padding_mask)
            
        return self.norm(X)
    

class DecoderLayer(nn.Module):
    def __init__(self, d_model, h, dropout_p=0.1):
        super().__init__()
        self.masked_mha = MultiHeadAttention(d_model, h, dropout_p)
        self.norm1 = LayerNorm(d_model)
        self.cross_mha = MultiHeadAttention(d_model, h, dropout_p)
        self.norm2 = LayerNorm(d_model)
        self.ff = FeedForwardLayers(d_model, dropout_p)
        self.norm3 = LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout_p)
        
    def forward(self, X, X_enc, enc_mask=None, dec_mask=None, cache=False):
        """
        X:        [Batch, Seq_len1, d_model]
        X_enc:    [Batch, Seq_len2, d_model]
        enc_mask: [Batch, 1, Seq_len2]
        dec_mask: [Batch, Seq_len1, Seq_len1]
        """
        norm_out = self.norm1(X)                                 # [Batch, Seq_len1, d_model]
        out = self.masked_mha(norm_out,                          # [Batch, Seq_len1, d_model]
                              norm_out, 
                              norm_out, dec_mask, cache=cache)       
        out = self.dropout(out)
        out = out + X

        norm_out = self.norm2(out)                               # [Batch, Seq_len1, d_model]
        cross_mha_out = self.cross_mha(norm_out,                 # [Batch, Seq_len1, d_model]
                                       X_enc, 
                                       X_enc, enc_mask, cache=False)    
        cross_mha_out = self.dropout(cross_mha_out)
        out = cross_mha_out + out

        norm_out = self.norm3(out)                               # [Batch, Seq_len1, d_model]
        ff_out = self.ff(norm_out)                               # [Batch, Seq_len1, d_model]
        ff_out = self.dropout(ff_out)
        out = ff_out + out
        
        return out

    def clear_cache(self):
        self.masked_mha.clear_cache()
    

class Decoder(nn.Module):
    def __init__(self, d_model, h, num_layers, dropout_p=0.1):
        super().__init__()
        self.num_layers = num_layers
        self.layers = nn.ModuleList(
            DecoderLayer(d_model, h, dropout_p) for _ in range(num_layers)
        )
        self.norm = LayerNorm(d_model)
        
    def forward(self, X, X_enc, enc_mask=None, dec_mask=None, cache=False):
        """
        X:        [Batch, Seq_len1, d_model]
        X_enc:    [Batch, Seq_len2, d_model]
        enc_mask: [Batch, 1, Seq_len2]
        dec_mask: [Batch, Seq_len1, Seq_len1]
        """
        for layer in self.layers:
            X = layer(X, X_enc, enc_mask, dec_mask, cache=cache)
            
        return self.norm(X)

    def clear_cache(self):
        for layer in self.layers: 
            layer.clear_cache()
            
    
class Transformer(nn.Module):
    def __init__(self, d_model, h, enc_num_layers, dec_num_layers, vocab_size, max_seq_len, dropout_p=0.1):
        super().__init__()
        self.enc_num_layers = enc_num_layers
        self.dec_num_layers = dec_num_layers
        self.d_model = d_model
        self.max_seq_len = max_seq_len
        self.pe = PositionalEncoding(d_model, max_seq_len)
        self.encoder = Encoder(d_model, h, enc_num_layers, dropout_p)
        self.decoder = Decoder(d_model, h, dec_num_layers, dropout_p)
        self.dropout = nn.Dropout(dropout_p)
        self.embedding = Embedding(vocab_size, d_model)
        nn.init.xavier_normal_(self.embedding.embedding.weight)
        self.projection_layer = nn.Linear(d_model, vocab_size)
        self.projection_layer.weight = self.embedding.embedding.weight
        
    def forward(self, X_enc, X_dec, enc_mask, dec_mask):
        """
        X_enc:      [Batch, Seq_len1]
        X_dec:      [Batch, Seq_len2]
        enc_mask:   [Batch, 1, Seq_len1]
        dec_mask:   [Batch, Seq_len2, Seq_len2]
        """
        X_enc = self.embedding(X_enc)                       # [Batch, Seq_len1, d_model]
        X_enc = self.pe(X_enc)                              # [Batch, Seq_len1, d_model]
        X_enc = self.dropout(X_enc)
        
        X_dec = self.embedding(X_dec)                       # [Batch, Seq_len2, d_model]
        X_dec = self.pe(X_dec)                              # [Batch, Seq_len2, d_model]
        X_dec = self.dropout(X_dec)
        
        out = self.encoder(X_enc, enc_mask)                 # [Batch, Seq_len1, d_model]
        out = self.decoder(X_dec, out, enc_mask, dec_mask)  # [Batch, Seq_len2, d_model]
        out = self.projection_layer(out)                    # [Batch, Seq_len2, vocab_size]
        
        return out

    def generate(self, X_enc, sos_id=1, eos_id=2, max_len=100, repetition_penalty=1.3, penalty_window=10):
        """
        X_enc: [Batch, Seq_len]
        """
        dec_id = torch.tensor([[sos_id]]).long().to(X_enc.device)
        
        X_enc_emb = self.embedding(X_enc)
        X_enc_emb = self.pe(X_enc_emb)
        enc_out = self.encoder(X_enc_emb)

        result = []
        
        i = 0
        while i < max_len:
            X_dec = self.embedding(dec_id)
            X_dec = X_dec + self.pe(i)
            out = self.decoder(X_dec, enc_out, cache=True) 
            logits = self.projection_layer(out)[0, -1, :] 
            
            # Определяем начало окна
            start_index = 0
            if penalty_window is not None and len(result) > penalty_window:
                start_index = len(result) - penalty_window
            
            # Берем только последние N токенов для проверки
            tokens_to_penalize = set(result[start_index:])
            
            for token in tokens_to_penalize:
                if logits[token] > 0:
                    logits[token] /= repetition_penalty
                else:
                    logits[token] *= repetition_penalty

            probs = torch.softmax(logits, dim=-1)
            next_token = torch.argmax(probs).item()
            
            result.append(next_token)
            
            if next_token == eos_id:
                break
                
            dec_id = torch.tensor([[next_token]]).long().to(X_enc.device)
            i += 1

        self.decoder.clear_cache()
        return result

In [4]:
@dataclass 
class Parameters:
    train_path: str = "gazeta-dataset/train.csv"
    valid_path: str = "gazeta-dataset/valid.csv"
    test_path: str =  "gazeta-dataset/test.csv"
    tokenizer_path: str = "summarization-tokenizer/transformer_tokenizer.json"
    synonyms_path: str = "synonyms-dict/synonyms_dict.json"
    batch_size: int = 4
    num_epochs: int = 30
    vocab_size: int = 30000
    d_model: int = 512
    h: int = 8
    enc_num_layers: int = 4
    dec_num_layers: int = 4
    max_seq_len: int = 5000

### Классы для аугментации

In [5]:
class FastSynonymAug:
    def __init__(self, dict_path="synonyms_dict.json", aug_p=0.4):
        self.aug_p = aug_p
        with open(dict_path, 'r', encoding='utf-8') as f:
            self.synonyms = json.load(f)

    def __call__(self, text):
        if not text: return text
        
        words = text.split()
        new_words = [
            self._replace_word(w) if random.random() < self.aug_p else w 
            for w in words
        ]
        
        return " ".join(new_words)

    def _replace_word(self, word):
        clean_word = word.lower().strip('.,!?:;') 

        if clean_word in self.synonyms:
            syn = random.choice(self.synonyms[clean_word])
            if word[0].isupper():
                syn = syn.capitalize()
            if not word[-1].isalnum():
                syn += word[-1]
            return syn
        return word


class FastWordSwapAug:
    def __init__(self, aug_p=0.15):
        self.aug_p = aug_p

    def __call__(self, text):
        words = text.split()
        length = len(words)
        
        if length < 2: return text
        
        if random.random() < self.aug_p:
            idx = random.randint(0, length-2)
            words[idx], words[idx+1] = words[idx+1], words[idx]
            return " ".join(words)
            
        return text


class EfficientPipeline:
    def __init__(self, synonyms_path):
        self.aug_synonym = FastSynonymAug(synonyms_path, aug_p=0.1)
        self.aug_swap = FastWordSwapAug(aug_p=0.25)
        
    def __call__(self, text, p=[0.7, 0.6]):
        if random.random() < p[0]:
            text = self.aug_synonym(text)

        if random.random() < p[1]:
            text = self.aug_swap(text)
            
        return text

In [6]:
class TextSummarizationDataset(Dataset):
    def __init__(self, data, tokenizer, augment=True):
        self.data = data
        self.tokenizer = tokenizer
        self.augment = augment

        if augment:
            self.augs = EfficientPipeline(Parameters.synonyms_path)

    def __getitem__(self, indx):
        data = self.data.iloc[indx]
        text, summary = data.iloc[0], data.iloc[1]

        if self.augment:
            text = self.augs(text)

        text_ids = self.tokenizer.encode(text).ids
        summary_ids = self.tokenizer.encode(summary).ids

        return torch.tensor(text_ids), torch.tensor(summary_ids)

    def __len__(self):
        return len(self.data)

### Инициализация объектов датасета

In [7]:
tokenizer = Tokenizer.from_file(Parameters.tokenizer_path)
train_data = pd.read_csv(Parameters.train_path)
valid_data = pd.read_csv(Parameters.valid_path)
test_data = pd.read_csv(Parameters.test_path)
train_data = pd.concat([train_data, test_data]).reset_index(drop=True)

In [8]:
train_dataset = TextSummarizationDataset(train_data, tokenizer)
valid_dataset = TextSummarizationDataset(valid_data, tokenizer, augment=False)

In [9]:
len(train_dataset), len(valid_dataset)

(67757, 6369)

In [10]:
def collate_fn(batch):
    text, summary = zip(*batch)

    pad_text = pad_sequence(text, batch_first=True, padding_value=0)
    pad_summary = pad_sequence(summary, batch_first=True, padding_value=0)
    
    return pad_text, pad_summary


def get_encoder_decoder_masks(pad_text, pad_summary, device):
    enc_mask = (pad_text != 0).unsqueeze(1).int()           # [Batch, 1, max_enc_seq_len]
    dec_pad_mask = (pad_summary != 0).unsqueeze(1)          # [Batch, 1, max_dec_seq_len]
    seq_len = dec_pad_mask.shape[-1]
    dec_triu_mask = torch.triu(                             # [1, max_dec_seq_len, max_dec_seq_len]
                        torch.ones(
                            (1, seq_len, seq_len), device=device
                        ), diagonal=1).int() == 0
    dec_mask = (dec_pad_mask & dec_triu_mask).int()         # [Batch, max_dec_seq_len, max_dec_seq_len]
    
    return enc_mask, dec_mask

In [11]:
train_loader = DataLoader(
    train_dataset, 
    batch_size=Parameters.batch_size, 
    shuffle=True, 
    num_workers=4, 
    drop_last=True, 
    collate_fn=collate_fn, 
    pin_memory=True,
    persistent_workers=True
)

valid_loader = DataLoader(
    valid_dataset, 
    batch_size=1, 
    shuffle=False, 
    num_workers=4, 
    drop_last=False, 
    collate_fn=collate_fn,
    pin_memory=True,
    persistent_workers=True
)

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [13]:
model = Transformer(
    d_model=Parameters.d_model, 
    h=Parameters.h, 
    enc_num_layers=Parameters.enc_num_layers, 
    dec_num_layers=Parameters.dec_num_layers, 
    vocab_size=Parameters.vocab_size, 
    max_seq_len=Parameters.max_seq_len,
    dropout_p=0.25
)

In [14]:
model = model.to(device)

Убирает `weight_decay` (L2-регуляризацию) со слоев нормализации (LayerNorm) и смещении (bias).

In [15]:
def get_grouped_params(model, weight_decay):
    decay_params = []
    no_decay_params = []
    
    for name, param in model.named_parameters():
        if "bias" in name or "embedding" in name or "norm" in name:
            no_decay_params.append(param)
        else:
            decay_params.append(param)
            
    return [
        {"params": decay_params, "weight_decay": weight_decay},
        {"params": no_decay_params, "weight_decay": 0.0},
    ]

In [16]:
train_loss = []
valid_metric = []
best_model = None
best_metric = 0
accumulation_steps = 6

In [17]:
pad_id = tokenizer.token_to_id("<PAD>")
sos_id = tokenizer.token_to_id("<SOS>")
eos_id = tokenizer.token_to_id("<EOS>")

In [18]:
grouped_params = get_grouped_params(model, weight_decay=5e-2)
optimizer = optim.AdamW(grouped_params, lr=7e-5)
criterion = nn.CrossEntropyLoss(ignore_index=0, label_smoothing=0.05)
rouge = ROUGEScore()
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, 
    max_lr=3e-4,
    steps_per_epoch=len(train_loader) // accumulation_steps, 
    epochs=Parameters.num_epochs,
    pct_start=0.1 
)

### Функция для валидации модели

In [33]:
def validation(model, valid_loader, penalty=1.3, window=10):
    model.eval() 
    rouge.reset()
    
    with torch.no_grad(): 
        for text, summary in tqdm(valid_loader):
            text = text.to(device)
            summary = summary[0].tolist()
            
            pred = model.generate(text, max_len=75, repetition_penalty=penalty, penalty_window=window)
            
            pred_str = tokenizer.decode(pred, skip_special_tokens=True)
            target_str = tokenizer.decode(summary, skip_special_tokens=True)
            
            rouge.update([pred_str], [target_str])
                        
    result = rouge.compute()

    return result

## Дообучение модели

In [ ]:
for epoch in range(Parameters.num_epochs):
    sum_loss = 0
    k = 0
    model.train()
    for i, (text, summary) in enumerate(tqdm(train_loader)):
        text = text.to(device)
        input_summary = summary[:, :-1].to(device)
        target_summary = summary[:, 1:].to(device)
        enc_mask, dec_mask = get_encoder_decoder_masks(text, input_summary, device)

        output = model(text, input_summary, enc_mask, dec_mask)
        pred = output.reshape(-1, output.shape[-1])
        target = target_summary.reshape(-1)
        loss = criterion(pred, target)

        sum_loss += loss.item()
        k += 1

        loss = loss / accumulation_steps
        loss.backward()

        if (i + 1) % accumulation_steps == 0:
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

    train_loss.append(sum_loss / k)
    print(f"Epoch {epoch + 1}; Loss: {train_loss[-1]:.4f}")
    
    model.eval() 
    rouge.reset() 

    if epoch > 5:   
        result = validation(model, valid_loader)
        rouge1 = result['rouge1_fmeasure'].item()
        rouge2 = result['rouge2_fmeasure'].item()
        rougeL = result['rougeL_fmeasure'].item()
        
        print(f"ROUGE-1: {rouge1:.4f} (совпадение слов)")
        print(f"ROUGE-2: {rouge2:.4f} (совпадение пар слов)")
        print(f"ROUGE-L: {rougeL:.4f} (структура предложений)")
    
        valid_metric.append(rouge2)
    
        if valid_metric[-1] > best_metric:
            best_metric = valid_metric[-1]
            best_model = deepcopy(model.state_dict())

        model_name = f"{epoch + 1}. Loss: {train_loss[-1]:.4f}, ROUGE-2: {rouge2:.4f}, ROUGE-L: {rougeL:.4f}.pth"
        torch.save(model.state_dict(), model_name)

In [20]:
weights = torch.load("best_model.pth", map_location="cuda")
model.load_state_dict(weights)

<All keys matched successfully>

### Подбор параметров `repetition_penalty` и `penalty_window`

In [22]:
for window in [25]:
    for penalty in [1.1, 1.15, 1.2, 1.25]:
        result = validation(model, valid_loader, penalty, window)

        rouge1 = result['rouge1_fmeasure'].item()
        rouge2 = result['rouge2_fmeasure'].item()
        rougeL = result['rougeL_fmeasure'].item()

        print(f"penalty = {penalty}")
        print(f"window = {window}")
        print(f"ROUGE-1: {rouge1:.4f} (совпадение слов)")
        print(f"ROUGE-2: {rouge2:.4f} (совпадение пар слов)")
        print(f"ROUGE-L: {rougeL:.4f} (структура предложений)")
        print("---" * 5)

  0%|          | 0/6369 [00:00<?, ?it/s]

penalty = 1.1
window = 25
ROUGE-1: 0.2021 (совпадение слов)
ROUGE-2: 0.0606 (совпадение пар слов)
ROUGE-L: 0.1977 (структура предложений)
---------------


  0%|          | 0/6369 [00:00<?, ?it/s]

penalty = 1.15
window = 25
ROUGE-1: 0.2039 (совпадение слов)
ROUGE-2: 0.0611 (совпадение пар слов)
ROUGE-L: 0.1994 (структура предложений)
---------------


  0%|          | 0/6369 [00:00<?, ?it/s]

penalty = 1.2
window = 25
ROUGE-1: 0.2043 (совпадение слов)
ROUGE-2: 0.0620 (совпадение пар слов)
ROUGE-L: 0.1999 (структура предложений)
---------------


  0%|          | 0/6369 [00:00<?, ?it/s]

penalty = 1.25
window = 25
ROUGE-1: 0.2052 (совпадение слов)
ROUGE-2: 0.0624 (совпадение пар слов)
ROUGE-L: 0.2009 (структура предложений)
---------------


## Примеры работы нейросети

In [55]:
def generate_summary(model, tokenizer, text):
    input_tensor = torch.tensor([tokenizer.encode(text).ids]).to(device)
    with torch.no_grad():
        pred_ids = model.generate(input_tensor, max_len=75, repetition_penalty=1.25, penalty_window=25)

    return tokenizer.decode(pred_ids)

In [56]:
model.eval();

## Тесты на новостных текстах

In [100]:
text = (
    "Вчера в Брюсселе завершился трехдневный саммит «Global AI Summit 2026», на котором "
    "представители 45 стран, включая лидеров технологического сектора и правозащитников, "
    "подписали «Декларацию цифрового суверенитета». Основная цель документа — создание единого "
    "правового поля для обучения нейросетей на авторском контенте. Согласно новым правилам, "
    "технологические гиганты, такие как NeuralSync и DeepMind, будут обязаны отчислять 0,5% от "
    "своей прибыли в фонд поддержки независимых авторов, чьи данные использовались для тренировки "
    "моделей. Генеральный директор NeuralSync Марк Вэнс в своем выступлении подчеркнул, что такие "
    "меры могут замедлить темпы инноваций. «Мы рискуем создать бюрократический барьер, который "
    "отбросит развитие ИИ на десятилетие назад», — заявил он. Тем не менее, комиссар ЕС по цифровой "
    "экономике Хелена Родригес парировала, отметив, что «без защиты интеллектуальной собственности "
    "сама концепция творчества потеряет смысл в эпоху машин». Помимо финансовых вопросов, декларация "
    "вводит обязательную маркировку любого контента (фото, видео, текст), созданного ИИ более чем на "
    "70%. На внедрение системы мониторинга странам-участницам дается 18 месяцев. Нарушение этих норм "
    "грозит штрафами в размере до 4% от годового оборота компании. Аналитики полагают, что это "
    "решение приведет к росту цен на подписки для конечных пользователей уже к началу следующего квартала."
)

In [101]:
print(f"ВЫВОД МОДЕЛИ: {generate_summary(model, tokenizer, text)}")

ВЫВОД МОДЕЛИ: в брюсселе завершился трехдневный саммит «global ai summit 2026», на котором представители 45 стран, включая лидеров технологического сектора и правозащитников, подписали «декларацию цифрового суверенитета». основная цель документа — создание единого правового поля для обучения нейросетей на авторском контенте.


In [102]:
text = (
    "Новое исследование, опубликованное в журнале Nature Climate Change, указывает на то, что система "
    "Атлантической меридиональной опрокидывающей циркуляции (AMOC), в которую входит Гольфстрим, теряет "
    "стабильность быстрее, чем предполагалось ранее. Причиной стало аномальное таяние ледников Гренландии, "
    "которое выбрасывает в океан гигантские объемы пресной воды, нарушая солевой баланс, необходимый для "
    "движения теплых потоков на север. Если текущие темпы потепления сохранятся, AMOC может полностью "
    "остановиться к 2040 году. Это приведет к катастрофическим последствиям для климата Европы и Северной "
    "Америки. В Великобритании и Скандинавии средняя зимняя температура может упасть на 10–15 градусов "
    "Цельсия, в то время как в тропических регионах усилится засуха и частота разрушительных ураганов. "
    "Доктор Эммануэль Жиро из Института климатических исследований в Потсдаме утверждает: «Это не просто "
    "прогноз, это физическая неизбежность, если мы не сократим выбросы метана и CO2 на 60% в ближайшие "
    "пять лет». В то же время некоторые климатологи-скептики указывают на естественные циклы океана, "
    "утверждая, что компьютерные модели могут переоценивать влияние пресной воды. Однако большинство "
    "правительств стран G7 уже начали разработку экстренных планов адаптации сельского хозяйства к возможным "
    "резким похолоданиям."
)

In [103]:
print(f"ВЫВОД МОДЕЛИ: {generate_summary(model, tokenizer, text)}")

ВЫВОД МОДЕЛИ: аномальное таяние ледников гренландии, которое выбрасывает в океан гигантские объемы пресной воды. это может привести к катастрофическим последствиям для европы и северной америки.


## Тесты `не` на новостных текстах

Видно, что модель начинает хуже генерировать текст, так как она обучалась преимущественно на текстах новостных статей.

In [104]:
text = (
    "Старый капитан долго смотрел на горизонт, где небо сливалось с тяжелыми свинцовыми волнами. " 
    "Его рука привычно сжимала потертую трубку, которая давно погасла. Он помнил, как сорок лет назад "
    "в этой же бухте он прощался с отцом, уходя в свое первое плавание. Тогда море казалось ему "
    "бесконечным полем возможностей, а сейчас — лишь кладбищем несбывшихся надежд. Город за его спиной "
    "шумел, жил своей суетливой жизнью, но капитан слышал только глухой рокот прибоя, который звал его " 
    "обратно, туда, где нет имен, а есть только соль и ветер."
)

In [105]:
print(f"ВЫВОД МОДЕЛИ: {generate_summary(model, tokenizer, text)}")

ВЫВОД МОДЕЛИ: в бухте он прощается с отцом, где нет ни зваля, то есть только соль и глухой рокот, но и туда, а также в этом году — на горизонте он живет с тяжелыми свинцовыми волнами.


In [106]:
text = (
    "Вчера в 14:00 по местному времени в центральном парке города N состоялось открытие новой "
    "библиотеки под открытым небом. На мероприятии присутствовали мэр города, инициативная группа "
    "волонтеров и более пятисот жителей. Библиотека включает в себя 12 стеллажей с книгами, зону "
    "для чтения с гамаками и точку бесплатного Wi-Fi. Проект был реализован на грантовые средства "
    "в размере 2 миллионов рублей. В планах организаторов — проводить здесь еженедельные литературные "
    "чтения и мастер-классы для детей. Мэр отметил, что это первый подобный объект в регионе, и если "
    "опыт будет успешным, такие зоны появятся в каждом районе."
)

In [107]:
print(f"ВЫВОД МОДЕЛИ: {generate_summary(model, tokenizer, text)}")

ВЫВОД МОДЕЛИ: в центральном парке n состоялось открытие новой библиотеки под открытым небом. на мероприятии приняли участие 12 стеллажей с гамаками wi-fi, в том числе и более пятисот жителей.


In [108]:
text = (
    "В случае возникновения форс-мажорных обстоятельств, включая, но не ограничиваясь: стихийные бедствия, "
    "военные действия или изменение законодательства, стороны освобождаются от ответственности за частичное "
    "или полное неисполнение обязательств по настоящему Договору, если указанные обстоятельства непосредственно " 
    "повлияли на исполнение Договора в установленный срок."
)

In [109]:
print(f"ВЫВОД МОДЕЛИ: {generate_summary(model, tokenizer, text)}")

ВЫВОД МОДЕЛИ: стихийные бедствия, военные и народные бедствия, освобожденные от ответственности за неисполнение обязательств по договору, вступили в силу с 1 июля текущего года.
